# ⚡ Image Reconstruction Under Corruption — SPEED-OPTIMIZED
### All optimizations applied for ~5–6 hour training target:
- **AMP** (Automatic Mixed Precision) — ~1.5–2× speedup
- **DataParallel** — uses both T4 GPUs if available
- **torch.compile** — fuses ops for faster execution
- **IMG_SIZE 128** — 4× fewer pixels than 256, huge speedup with minimal quality loss
- **Larger batch size** — better GPU utilization
- **num_workers=4, pin_memory=True, persistent_workers=True** — zero CPU bottleneck
- **Prefetch factor** — loads next batch in background
- **Checkpointing every 5 epochs** — resume safely after Kaggle timeout
- **Full tqdm progress** with ETA on every batch

In [1]:
import os, time, csv, glob
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}:', torch.cuda.get_device_name(i))

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [2]:
# ── Paths ──
BASE         = '/kaggle/input/competitions/image-reconstruction-under-corruption/Datasets'
TRAIN_CLEAN  = os.path.join(BASE, 'train_clean')
TRAIN_CORRUPT= os.path.join(BASE, 'train_corrupt')
TEST_CORRUPT = os.path.join(BASE, 'test_corrupt')
WORK_DIR     = '/kaggle/working'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('Using device:', DEVICE, '| GPUs:', N_GPUS)

Using device: cuda | GPUs: 2


In [3]:
# ── Peek at data ──
sample_img = Image.open(os.path.join(TRAIN_CLEAN, sorted(os.listdir(TRAIN_CLEAN))[0]))
print('Image size:', sample_img.size, '| Mode:', sample_img.mode)

n_train = len(os.listdir(TRAIN_CLEAN))
n_test  = len(os.listdir(TEST_CORRUPT))
print(f'Train pairs: {n_train} | Test images: {n_test}')

Image size: (32, 32) | Mode: RGB
Train pairs: 48000 | Test images: 12000


In [4]:
# ══════════════════════════════════════════════════
# ⚙️  CONFIG
# ══════════════════════════════════════════════════

IMG_SIZE   = 128
BATCH_SIZE = 64 if N_GPUS <= 1 else 128
EPOCHS     = 60
LR         = 2e-4
VAL_SPLIT  = 0.1
SEED       = 42

CHECKPOINT_EVERY = 5
RESUME_FROM      = None

NUM_WORKERS = 4
PREFETCH    = 2

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Config: IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}')
print(f'Checkpointing every {CHECKPOINT_EVERY} epochs')

Config: IMG_SIZE=128, BATCH_SIZE=128, EPOCHS=60
Checkpointing every 5 epochs


In [5]:
# ── Dataset ──

class ReconstructionDataset(Dataset):
    def __init__(self, corrupt_dir, clean_dir=None, img_size=128, augment=False):
        self.corrupt_paths = sorted(glob.glob(os.path.join(corrupt_dir, '*.png')))
        self.clean_dir  = clean_dir
        self.img_size   = img_size
        self.augment    = augment

    def __len__(self):
        return len(self.corrupt_paths)

    def __getitem__(self, idx):
        cpath = self.corrupt_paths[idx]
        fname = os.path.basename(cpath)

        corrupt_img = Image.open(cpath).convert('RGB')
        corrupt_img = corrupt_img.resize((self.img_size, self.img_size), Image.BICUBIC)

        if self.clean_dir is not None:
            clean_path = os.path.join(self.clean_dir, fname)
            clean_img  = Image.open(clean_path).convert('RGB')
            clean_img  = clean_img.resize((self.img_size, self.img_size), Image.BICUBIC)

            if self.augment:
                if np.random.rand() > 0.5:
                    corrupt_img = TF.hflip(corrupt_img)
                    clean_img   = TF.hflip(clean_img)
                if np.random.rand() > 0.5:
                    corrupt_img = TF.vflip(corrupt_img)
                    clean_img   = TF.vflip(clean_img)
                angle = int(np.random.choice([0, 90, 180, 270]))
                if angle != 0:
                    corrupt_img = TF.rotate(corrupt_img, angle)
                    clean_img   = TF.rotate(clean_img, angle)

            return TF.to_tensor(corrupt_img), TF.to_tensor(clean_img)
        else:
            return TF.to_tensor(corrupt_img), fname


class SplitDataset(ReconstructionDataset):
    def __init__(self, file_set, corrupt_dir, clean_dir, img_size, augment):
        super().__init__(corrupt_dir, clean_dir, img_size, augment)
        self.corrupt_paths = [p for p in self.corrupt_paths
                              if os.path.basename(p) in file_set]


all_files = sorted(os.listdir(TRAIN_CORRUPT))
np.random.shuffle(all_files)
n_val       = max(1, int(len(all_files) * VAL_SPLIT))
val_files   = set(all_files[:n_val])
train_files = set(all_files[n_val:])

train_ds = SplitDataset(train_files, TRAIN_CORRUPT, TRAIN_CLEAN, IMG_SIZE, augment=True)
val_ds   = SplitDataset(val_files,   TRAIN_CORRUPT, TRAIN_CLEAN, IMG_SIZE, augment=False)
test_ds  = ReconstructionDataset(TEST_CORRUPT, clean_dir=None, img_size=IMG_SIZE)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=PREFETCH
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=PREFETCH
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=PREFETCH
)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Train: 43200 | Val: 4800 | Test: 12000
Train batches: 338 | Val batches: 38


In [6]:
# ── U-Net Architecture ──

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, base=64):
        super().__init__()
        self.enc1 = ConvBlock(in_ch,   base)
        self.enc2 = ConvBlock(base,    base*2)
        self.enc3 = ConvBlock(base*2,  base*4)
        self.enc4 = ConvBlock(base*4,  base*8)
        self.bottleneck = ConvBlock(base*8, base*16, dropout=0.3)
        self.up4   = nn.ConvTranspose2d(base*16, base*8,  2, stride=2)
        self.dec4  = ConvBlock(base*16, base*8)
        self.up3   = nn.ConvTranspose2d(base*8,  base*4,  2, stride=2)
        self.dec3  = ConvBlock(base*8,  base*4)
        self.up2   = nn.ConvTranspose2d(base*4,  base*2,  2, stride=2)
        self.dec2  = ConvBlock(base*4,  base*2)
        self.up1   = nn.ConvTranspose2d(base*2,  base,    2, stride=2)
        self.dec1  = ConvBlock(base*2,  base)
        self.pool  = nn.MaxPool2d(2)
        self.final = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        # sigmoid → [0, 1] float; to_pil_image will convert to uint8 [0, 255]
        return torch.sigmoid(self.final(d1))


model = UNet(base=64).to(DEVICE)

if N_GPUS > 1:
    print(f'Using DataParallel on {N_GPUS} GPUs')
    model = nn.DataParallel(model)

try:
    model = torch.compile(model, mode='reduce-overhead')
    print('torch.compile: enabled (reduce-overhead mode)')
except Exception as e:
    print(f'torch.compile not available ({e}), skipping')

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {params:,}')

Using DataParallel on 2 GPUs
torch.compile: enabled (reduce-overhead mode)
Trainable parameters: 31,037,763


In [7]:
# ── Loss: MSE (80%) + L1 (20%) ──

def combined_loss(pred, target, mse_w=0.8, l1_w=0.2):
    mse = nn.functional.mse_loss(pred, target)
    l1  = nn.functional.l1_loss(pred, target)
    return mse_w * mse + l1_w * l1


optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = GradScaler()

print('Optimizer: AdamW | Scheduler: CosineAnnealing | AMP: enabled')

Optimizer: AdamW | Scheduler: CosineAnnealing | AMP: enabled


/tmp/ipykernel_23/3200929157.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()


In [8]:
# ── Checkpoint utilities ──

def _get_raw_model(model):
    m = model
    if hasattr(m, 'module'):    m = m.module
    if hasattr(m, '_orig_mod'): m = m._orig_mod
    return m


def save_checkpoint(epoch, model, optimizer, scheduler, scaler, best_val_mse, path):
    torch.save({
        'epoch':        epoch,
        'model_state':  _get_raw_model(model).state_dict(),
        'optimizer':    optimizer.state_dict(),
        'scheduler':    scheduler.state_dict(),
        'scaler':       scaler.state_dict(),
        'best_val_mse': best_val_mse,
    }, path)
    print(f'  💾 Checkpoint saved: {path}')


def load_checkpoint(path, model, optimizer, scheduler, scaler):
    ckpt = torch.load(path, map_location=DEVICE)
    _get_raw_model(model).load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    print(f'  ✅ Resumed from epoch {ckpt["epoch"]} | Best val MSE: {ckpt["best_val_mse"]:.6f}')
    return ckpt['epoch'] + 1, ckpt['best_val_mse']


start_epoch  = 1
best_val_mse = float('inf')

if RESUME_FROM and os.path.exists(RESUME_FROM):
    start_epoch, best_val_mse = load_checkpoint(
        RESUME_FROM, model, optimizer, scheduler, scaler
    )
else:
    print('Starting fresh training from epoch 1')

Starting fresh training from epoch 1


In [9]:
# ══════════════════════════════════════════════════
# 🚀 TRAINING LOOP
# ══════════════════════════════════════════════════

train_start = time.time()
history = []

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.time()

    model.train()
    train_loss = 0.0
    train_n    = 0

    train_bar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]',
                     leave=True, dynamic_ncols=True)

    for corrupt, clean in train_bar:
        corrupt = corrupt.to(DEVICE, non_blocking=True)
        clean   = clean.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            pred = model(corrupt)
            loss = combined_loss(pred, clean)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        bs          = corrupt.size(0)
        train_loss += loss.item() * bs
        train_n    += bs
        train_bar.set_postfix({'loss': f'{loss.item():.5f}'})

    avg_train_loss = train_loss / train_n

    model.eval()
    val_mse = 0.0
    val_n   = 0

    val_bar = tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS} [Val]  ',
                   leave=False, dynamic_ncols=True)

    with torch.no_grad():
        for corrupt, clean in val_bar:
            corrupt = corrupt.to(DEVICE, non_blocking=True)
            clean   = clean.to(DEVICE, non_blocking=True)
            with autocast():
                pred = model(corrupt)
            mse      = nn.functional.mse_loss(pred.float(), clean.float()).item()
            bs       = corrupt.size(0)
            val_mse += mse * bs
            val_n   += bs
            val_bar.set_postfix({'mse': f'{mse:.5f}'})

    avg_val_mse = val_mse / val_n
    scheduler.step()

    epoch_secs       = time.time() - epoch_start
    remaining_epochs = EPOCHS - epoch
    eta_secs         = epoch_secs * remaining_epochs

    history.append({
        'epoch': epoch, 'train_loss': avg_train_loss,
        'val_mse': avg_val_mse, 'epoch_time_min': epoch_secs / 60
    })

    if avg_val_mse < best_val_mse:
        best_val_mse = avg_val_mse
        torch.save(_get_raw_model(model).state_dict(),
                   os.path.join(WORK_DIR, 'best_unet.pth'))
        star = ' ⭐ NEW BEST'
    else:
        star = ''

    print(f'Epoch {epoch:>3}/{EPOCHS} | '
          f'Train Loss: {avg_train_loss:.5f} | '
          f'Val MSE: {avg_val_mse:.5f} | '
          f'Best: {best_val_mse:.5f}{star} | '
          f'Time: {epoch_secs/60:.1f}m | '
          f'ETA: {eta_secs/3600:.1f}h | '
          f'LR: {scheduler.get_last_lr()[0]:.2e}')

    if epoch % CHECKPOINT_EVERY == 0:
        ckpt_path = os.path.join(WORK_DIR, f'checkpoint_epoch_{epoch:03d}.pth')
        save_checkpoint(epoch, model, optimizer, scheduler, scaler, best_val_mse, ckpt_path)
        for old_ep in range(epoch - 2 * CHECKPOINT_EVERY, epoch - CHECKPOINT_EVERY, CHECKPOINT_EVERY):
            old_path = os.path.join(WORK_DIR, f'checkpoint_epoch_{old_ep:03d}.pth')
            if os.path.exists(old_path):
                os.remove(old_path)
                print(f'  🗑️  Removed old checkpoint: {old_path}')

total_time = (time.time() - train_start) / 3600
print(f'\n✅ Training done! Total time: {total_time:.2f} hours')
print(f'Best Val MSE: {best_val_mse:.6f}')

Epoch 1/60 [Train]:   0%|          | 0/338 [00:00<?, ?it/s]

/tmp/ipykernel_23/1710101057.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
W0418 08:58:30.314000 23 torch/_logging/_internal.py:1204] [0/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/variables/functions.py:1946: UserWarning: Dynamo does not know how to trace the builtin `_thread.get_ident.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround.
If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use `torc

Epoch 1/60 [Val]  :   0%|          | 0/38 [00:00<?, ?it/s]

/tmp/ipykernel_23/1710101057.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_23/1710101057.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch   1/60 | Train Loss: 0.04985 | Val MSE: 0.02796 | Best: 0.02796 ⭐ NEW BEST | Time: 3.3m | ETA: 3.3h | LR: 2.00e-04


...

Epoch 59/60 [Val]  :   0%|          | 0/38 [00:00<?, ?it/s]

Epoch  59/60 | Train Loss: 0.02567 | Val MSE: 0.01507 | Best: 0.01507 | Time: 2.9m | ETA: 0.0h | LR: 1.14e-06


Epoch 60/60 [Train]:   0%|          | 0/338 [00:00<?, ?it/s]

Epoch 60/60 [Val]  :   0%|          | 0/38 [00:00<?, ?it/s]

Epoch  60/60 | Train Loss: 0.02565 | Val MSE: 0.01516 | Best: 0.01507 | Time: 2.9m | ETA: 0.0h | LR: 1.00e-06
  💾 Checkpoint saved: /kaggle/working/checkpoint_epoch_060.pth
  🗑️  Removed old checkpoint: /kaggle/working/checkpoint_epoch_050.pth

✅ Training done! Total time: 2.92 hours
Best Val MSE: 0.015068


In [10]:
history_df = pd.DataFrame(history)
print(history_df.to_string(index=False))
history_df.to_csv(os.path.join(WORK_DIR, 'training_history.csv'), index=False)
print('\nHistory saved.')

 epoch  train_loss  val_mse  epoch_time_min
     1    0.049854 0.027955        3.343017
     2    0.043670 0.024400        2.866959
     3    0.039855 0.023352        2.873623
     4    0.037816 0.021744        2.870193
     5    0.036700 0.020269        2.868281
     6    0.035770 0.019918        2.875299
     7    0.035082 0.019734        2.891926
     8    0.034495 0.018994        2.883092
     9    0.034069 0.019070        2.882260
    10    0.033501 0.019294        2.883436
    11    0.033189 0.018708        2.890003
    12    0.032769 0.017894        2.889783
    13    0.032460 0.018874        2.894826
    14    0.032167 0.017353        2.893179
    15    0.031935 0.017811        2.893071
    16    0.031641 0.017940        2.881699
    17    0.031350 0.017472        2.908467
    18    0.031083 0.017146        2.901649
    19    0.030917 0.017375        2.914736
    20    0.030630 0.017115        2.909273
    21    0.030492 0.017083        2.898256
    22    0.030262 0.016564     

In [11]:
# ── Load best model for inference ──

inference_model = UNet(base=64).to(DEVICE)
inference_model.load_state_dict(
    torch.load(os.path.join(WORK_DIR, 'best_unet.pth'), map_location=DEVICE)
)
if N_GPUS > 1:
    inference_model = nn.DataParallel(inference_model)
inference_model.eval()
print('✅ Best model loaded for inference.')

✅ Best model loaded for inference.


In [12]:
# ── Detect original test image size ──
test_files  = sorted(os.listdir(TEST_CORRUPT))
sample_test = Image.open(os.path.join(TEST_CORRUPT, test_files[0]))
ORIG_H, ORIG_W = sample_test.size[1], sample_test.size[0]
print(f'Original test image size: {ORIG_W}×{ORIG_H}')

Original test image size: 32×32


In [13]:
# ══════════════════════════════════════════════════
# 🔮 Inference with TTA (horizontal flip averaging)
# ══════════════════════════════════════════════════
#
# Value range note:
#   - Model output: sigmoid → [0.0, 1.0] float32
#   - clamp(0, 1) is a safety guard (sigmoid already guarantees this)
#   - TF.to_pil_image on a float [0,1] tensor → PIL RGB uint8 [0, 255]
#   - Resize (BICUBIC) keeps values in [0, 255] uint8
#   - np.array(pred_pil) → uint8 array, values 0–255  ✅
#   These integers are written directly into the submission CSV.

all_predictions = {}  # fname → numpy (H, W, 3) uint8, values 0–255

def predict_with_tta(model, img_tensor):
    """img_tensor: [C,H,W] on DEVICE. Returns averaged [C,H,W] float32 in [0,1]."""
    with torch.no_grad(), autocast():
        pred_orig = model(img_tensor.unsqueeze(0)).float()[0]
        pred_flip = model(TF.hflip(img_tensor).unsqueeze(0)).float()[0]
        pred_flip = TF.hflip(pred_flip)
    return (pred_orig + pred_flip) / 2.0


for corrupt_t, fnames in tqdm(test_loader, desc='🔮 Inference', dynamic_ncols=True):
    corrupt_t = corrupt_t.to(DEVICE, non_blocking=True)
    B = corrupt_t.size(0)
    for i in range(B):
        fname = fnames[i]
        pred  = predict_with_tta(inference_model, corrupt_t[i])

        # clamp is a safety guard; sigmoid already outputs [0,1]
        pred_pil = TF.to_pil_image(pred.cpu().clamp(0.0, 1.0))
        # Resize back to native 32×32 (or whatever ORIG size is)
        if (ORIG_W, ORIG_H) != (IMG_SIZE, IMG_SIZE):
            pred_pil = pred_pil.resize((ORIG_W, ORIG_H), Image.BICUBIC)

        # np.array of a PIL RGB image → uint8, shape (H,W,3), values 0–255
        all_predictions[fname] = np.array(pred_pil, dtype=np.uint8)

# Quick sanity check on the stored arrays
sample_arr = next(iter(all_predictions.values()))
print(f'Predictions ready for {len(all_predictions)} images.')
print(f'Array dtype: {sample_arr.dtype} | shape: {sample_arr.shape} | '
      f'min: {sample_arr.min()} | max: {sample_arr.max()}')  # expect uint8, 0–255

🔮 Inference:   0%|          | 0/94 [00:01<?, ?it/s]

/tmp/ipykernel_23/2653378569.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


Predictions ready for 12000 images.
Array dtype: uint8 | shape: (32, 32, 3) | min: 20 | max: 253


In [14]:
# ══════════════════════════════════════════════════
# 📄 Build submission CSV
# FORMAT: id, pixel_0, pixel_1, ..., pixel_{H*W*3 - 1}
# Each pixel value is a separate column, integer 0–255.
# Pixel order: row-major, channels R,G,B (numpy default flatten).
# ══════════════════════════════════════════════════

def build_submission(all_preds, out_path):
    """
    Creates submission.csv with columns:
      id, pixel_0, pixel_1, ..., pixel_{N-1}
    where N = H * W * 3 (one column per pixel channel value, integer 0–255).
    """
    sorted_fnames = sorted(all_preds.keys())

    # Derive pixel column count from the first prediction
    sample_flat = all_preds[sorted_fnames[0]].flatten()
    n_pixels    = len(sample_flat)  # 32*32*3 = 3072 for native images

    header = ['id'] + [f'pixel_{i}' for i in range(n_pixels)]
    rows   = [header]

    for fname in tqdm(sorted_fnames, desc='Building CSV', dynamic_ncols=True):
        img_arr = all_preds[fname]           # (H, W, 3) uint8, values 0–255
        flat    = img_arr.flatten().tolist() # list of ints 0–255

        # Strip extension: test_00000.png → test_00000
        img_id = os.path.splitext(fname)[0]
        rows.append([img_id] + flat)

    with open(out_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerows(rows)

    print(f'✅ Submission saved: {out_path}')
    print(f'   Rows (excl. header): {len(rows)-1}')
    print(f'   Columns: 1 (id) + {n_pixels} pixel cols = {len(header)} total')


build_submission(all_predictions, os.path.join(WORK_DIR, 'submission.csv'))

Building CSV:   0%|          | 0/12000 [00:00<?, ?it/s]

✅ Submission saved: /kaggle/working/submission.csv
   Rows (excl. header): 12000
   Columns: 1 (id) + 3072 pixel cols = 3073 total


In [15]:
# ── Sanity check ──
sub_df = pd.read_csv(os.path.join(WORK_DIR, 'submission.csv'), nrows=4)
print('Shape:', sub_df.shape)
print('First 10 columns:', sub_df.columns[:10].tolist())
print('Last 5 columns: ', sub_df.columns[-5:].tolist())
print()
print(sub_df.iloc[:, :10])  # id + first 9 pixel cols
print()

# Verify pixel value range
pixel_cols = sub_df.columns[1:]  # all except 'id'
print(f'Pixel value range — min: {sub_df[pixel_cols].min().min()}, '
      f'max: {sub_df[pixel_cols].max().max()}')  # should be 0–255

import subprocess
result = subprocess.run(
    ['wc', '-l', os.path.join(WORK_DIR, 'submission.csv')],
    capture_output=True, text=True
)
print('Total rows (incl header):', result.stdout.strip())
print('\n🎉 All done! Submit /kaggle/working/submission.csv')

Shape: (4, 3073)
First 10 columns: ['id', 'pixel_0', 'pixel_1', 'pixel_2', 'pixel_3', 'pixel_4', 'pixel_5', 'pixel_6', 'pixel_7', 'pixel_8']
Last 5 columns:  ['pixel_3067', 'pixel_3068', 'pixel_3069', 'pixel_3070', 'pixel_3071']

           id  pixel_0  pixel_1  pixel_2  pixel_3  pixel_4  pixel_5  pixel_6  \
0  test_00000      225      223      222      228      225      221      232   
1  test_00001      131      191      235      124      193      240      123   
2  test_00002      146      129       93      143      132       91      137   
3  test_00003       69       66       43       56       58       35       53   

   pixel_7  pixel_8  
0      228      223  
1      192      239  
2      131       89  
3       53       33  

Pixel value range — min: 9, max: 253
Total rows (incl header): 12001 /kaggle/working/submission.csv

🎉 All done! Submit /kaggle/working/submission.csv


In [16]:
import os
import glob
import shutil

WORK_DIR = '/kaggle/working'
KEEP_FILE = 'submission.csv'

print("Cleaning up workspace...")

# Iterate over all items in the working directory
for item in os.listdir(WORK_DIR):
    if item != KEEP_FILE:
        item_path = os.path.join(WORK_DIR, item)
        try:
            # Remove files
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.remove(item_path)
                print(f"🗑️ Removed file: {item}")
            # Remove directories (like .virtual_documents if needed)
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
                print(f"🗑️ Removed directory: {item}")
        except Exception as e:
            print(f"Failed to delete {item_path}. Reason: {e}")

print("\n✅ Cleanup complete. Remaining files in /kaggle/working:")
print(os.listdir(WORK_DIR))

Cleaning up workspace...
🗑️ Removed file: checkpoint_epoch_060.pth
🗑️ Removed file: training_history.csv
🗑️ Removed file: best_unet.pth
🗑️ Removed file: checkpoint_epoch_055.pth
🗑️ Removed file: __notebook__.ipynb

✅ Cleanup complete. Remaining files in /kaggle/working:
['submission.csv']
